# Notebook 2 — Acquiring Sentinel-1, Sentinel-2, Landsat
**Objectives**
- Build clean, cloud-filtered Sentinel-2 composites
- Build Sentinel-1 GRD composites with correct orbit/polarization selection
- Build Landsat 8/9 Collection-2 Surface Reflectance composites
- Compare the three sources side-by-side
- Export to Drive, to an Asset, and to local disk

All sections use the same working AOI (Lake Naivasha) so results are directly comparable.

In [ ]:
import ee, geemap
ee.Initialize(project="operating-rush-505912-s8")  # re-run if starting a fresh kernel

AOI = ee.Geometry.Rectangle([36.20, -0.83, 36.45, -0.65])  # Lake Naivasha
START, END = "2023-01-01", "2023-03-01"


## Section A — Sentinel-2 acquisition
### A.1 Cloud masking
Use the `QA60` bitmask band (fast, coarse) or the `s2cloudless` probability collection (slower, more accurate). We use `QA60` here for speed; swap in `s2cloudless` for production work.

In [ ]:
def mask_s2_clouds(image):
    qa = image.select('QA60')
    cloud_bit = 1 << 10
    cirrus_bit = 1 << 11
    mask = qa.bitwiseAnd(cloud_bit).eq(0).And(qa.bitwiseAnd(cirrus_bit).eq(0))
    return image.updateMask(mask).divide(10000).copyProperties(image, image.propertyNames())

s2_col = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
          .filterBounds(AOI)
          .filterDate(START, END)
          .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 40))
          .map(mask_s2_clouds))

print("S2 scenes after filter:", s2_col.size().getInfo())
s2_composite = s2_col.median().clip(AOI)

In [ ]:
Map = geemap.Map(center=[-0.74, 36.33], zoom=11)
Map.addLayer(s2_composite, {"bands": ["B4","B3","B2"], "min": 0, "max": 0.3}, "S2 true color")
Map.addLayer(s2_composite, {"bands": ["B8","B4","B3"], "min": 0, "max": 0.4}, "S2 false color (NIR)", shown=False)
Map

## Section B — Sentinel-1 acquisition
S1 GRD is already terrain-corrected and calibrated to sigma-nought (dB) by GEE. Filter on **instrument mode**, **polarization**, and **orbit pass** before compositing — mixing ascending and descending passes blurs directional backscatter effects.

In [ ]:
s1_col = (ee.ImageCollection("COPERNICUS/S1_GRD")
          .filterBounds(AOI)
          .filterDate(START, END)
          .filter(ee.Filter.eq("instrumentMode", "IW"))
          .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
          .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
          .filter(ee.Filter.eq("orbitProperties_pass", "ASCENDING"))
          .select(["VV", "VH"]))

print("S1 scenes after filter:", s1_col.size().getInfo())
s1_composite = s1_col.median().clip(AOI)

In [ ]:
Map.addLayer(s1_composite, {"bands": ["VV"], "min": -25, "max": 0}, "S1 VV (dB)", shown=False)
Map.addLayer(s1_composite, {"bands": ["VV","VH","VV"], "min": -25, "max": 0}, "S1 VV/VH/VV composite", shown=False)
Map

## Section C — Landsat acquisition (Landsat 8/9, Collection 2 Level-2 SR)
Collection-2 SR bands need the official scale/offset applied before use (`SR_B* * 0.0000275 - 0.2`). Cloud-mask with the `QA_PIXEL` bitmask.

In [ ]:
def mask_l8_clouds(image):
    qa = image.select('QA_PIXEL')
    cloud_bit, shadow_bit, cirrus_bit = 1 << 3, 1 << 4, 1 << 2
    mask = (qa.bitwiseAnd(cloud_bit).eq(0)
            .And(qa.bitwiseAnd(shadow_bit).eq(0))
            .And(qa.bitwiseAnd(cirrus_bit).eq(0)))
    optical = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    return image.addBands(optical, None, True).updateMask(mask)

l8_col = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
          .filterBounds(AOI)
          .filterDate(START, END)
          .filter(ee.Filter.lt("CLOUD_COVER", 40))
          .map(mask_l8_clouds))

print("Landsat 8 scenes after filter:", l8_col.size().getInfo())
l8_composite = l8_col.median().clip(AOI)

In [ ]:
Map.addLayer(l8_composite, {"bands": ["SR_B4","SR_B3","SR_B2"], "min": 0, "max": 0.3}, "Landsat 8 true color", shown=False)
Map

## Side-by-side comparison
`geemap.linked_maps` lets you pan/zoom all panels together — the fastest way to *see* the resolution and radiometric differences between sources.

In [ ]:
images  = [s2_composite, l8_composite, s1_composite]
vis_params = [
    {"bands": ["B4","B3","B2"], "min": 0, "max": 0.3},
    {"bands": ["SR_B4","SR_B3","SR_B2"], "min": 0, "max": 0.3},
    {"bands": ["VV"], "min": -25, "max": 0},
]
labels = ["Sentinel-2 (10 m)", "Landsat 8 (30 m)", "Sentinel-1 VV (10 m, SAR)"]

linked_map = geemap.linked_maps(
    rows=1, cols=3, height="350px",
    center=[-0.74, 36.33], zoom=11,
    ee_objects=images, vis_params=vis_params, labels=labels
)
linked_map

**look for:**
- Spatial detail: S2/S1 (10 m) resolve the lake shoreline and shore-side farms far better than Landsat (30 m)
- Radiometry: SAR (S1) responds to surface roughness/dielectric properties, not color — calm open water is very dark (low backscatter), which is the physical basis for SAR water masking in Notebook 4
- Revisit: S2 (5-day), Landsat (16-day), S1 (6–12-day depending on orbit) — affects how easy it is to get a cloud-free composite in each

## Export paths
Three destinations, three tools. Pick based on where the result needs to live next.

In [ ]:
# 1) Export to Google Drive — best for one-off large downloads you'll grab later
task_drive = ee.batch.Export.image.toDrive(
    image=s2_composite.select(["B4","B3","B2","B8"]),
    description="naivasha_s2_composite",
    folder="gee_exports",
    region=AOI,
    scale=10,
    crs="EPSG:32637",       # UTM 37N — correct zone for this AOI
    maxPixels=1e10
)
task_drive.start()
print("Drive export task started:", task_drive.status()['state'])

In [ ]:
# 2) Export to a GEE Asset — best when you'll re-use the result in later server-side steps
task_asset = ee.batch.Export.image.toAsset(
    image=s2_composite.select(["B4","B3","B2","B8"]),
    description="naivasha_s2_composite_asset",
    assetId="projects/YOUR-GEE-PROJECT-ID/assets/naivasha_s2_composite",
    region=AOI,
    scale=10,
    crs="EPSG:32637",
    maxPixels=1e10
)
task_asset.start()
print("Asset export task started:", task_asset.status()['state'])

In [ ]:
# 3) Export straight to local disk via geemap — best for small AOIs / quick iteration
geemap.ee_export_image(
    s2_composite.select(["B4","B3","B2","B8"]),
    filename="naivasha_s2_composite.tif",
    scale=10,
    region=AOI,
    crs="EPSG:32637",
    file_per_band=False
)
print("Local export complete (check working directory).")

**Common export errors:**
| Symptom | Usual cause | Fix |
|---|---|---|
| `Total request size exceeds ...` | Region + scale too large for a synchronous local export | Use `Export.image.toDrive`/`toAsset` instead, or coarsen `scale` |
| Export silently reprojects oddly | No explicit `crs` set | Always set `crs` explicitly (don't rely on the default) |
| `Image.clip: Parameter 'geometry' is required` | AOI geometry is `None` / not evaluated server-side yet | Make sure `AOI` is an `ee.Geometry`, not a client-side list |
| Task stuck in `READY` forever | Not polled / Earth Engine quota queue | Check the [Tasks tab](https://code.earthengine.google.com/tasks) in the Code Editor |

## Exercise 2.1 — Naivasha acquisition
**Task:** Build your own cloud-free composite over Lake Naivasha for a *different* time window, then export it.

1. Pick a 2-month window in the **dry season** (June–September) instead of the wet-season window used above
2. Repeat the Sentinel-2 cloud-masked composite for that window
3. Repeat the Sentinel-1 ascending-orbit VV/VH composite for the same window
4. Export both to Drive, in the same UTM 37N CRS, at native resolution
5. **Discuss:** how does scene count and cloud cover change between wet and dry season? Does the SAR composite need the season change at all — why/why not?

In [ ]:
# Exercise 2.1 scaffold: fill in the TODOs
EX_START, EX_END = "2023-07-01", "2023-09-01"  # TODO: confirm dry-season window

# TODO 1: build ex_s2_col the same way as s2_col above, but with EX_START/EX_END
ex_s2_col = None

# TODO 2: build ex_s1_col the same way as s1_col above, but with EX_START/EX_END
ex_s1_col = None

# TODO 3: composite both and print scene counts
# print("Dry-season S2 scenes:", ex_s2_col.size().getInfo())
# print("Dry-season S1 scenes:", ex_s1_col.size().getInfo())

# TODO 4: export both composites to Drive (reuse the toDrive pattern above)


**Walkthrough (uncover after attempting):**
- Dry-season S2 scene count is typically *higher* than wet-season for the same window length — fewer scenes get discarded by the `CLOUDY_PIXEL_PERCENTAGE` filter
- S1 scene count is largely season-independent — radar penetrates cloud, so the only variable is orbit repeat cycle, not weather
- This is exactly why production water-monitoring pipelines (see Day 3 wrap-up) lean on SAR for wet-season/flood monitoring and reserve optical indices for dry-season detail work